# Replication Starter: Cheng & Hoekstra (2013), Castle Doctrine Laws and Homicide
## ECON 5200 · Midterm Replication Study · difference-in-differences

Cheng, C. and Hoekstra, M. (2013). "Does Strengthening Self-Defense Law Deter Crime or Escalate Violence? Evidence
from Expansions to Castle Doctrine." *Journal of Human Resources* 48(3): 821–854.

> **AI rule for this project.** Sections 1 and 2 (positioning, direct replication) are **no AI**: not for
> the code, not for the prose. Sections 3 to 6 allow AI as a documented co-pilot under P.R.I.M.E., with the full
> transcript in `ai-appendix.md`. Every cell below is marked with the section it belongs to.

**How to use this notebook.** It is a starter, not the project. Copy it into your repository made from
[RyanPiao/econ5200-replication-template](https://github.com/RyanPiao/econ5200-replication-template), run it top to
bottom, and fill the cells marked **YOUR TURN**. Cells marked **GIVEN** run as they are; cells marked **CHECK** test
what you wrote and print a message, never the answer. A blank copy runs from top to bottom and prints no estimate:
the numbers appear once you have written Section 2.

In [ ]:
# GIVEN — setup
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

print("Setup complete.")

## The data

`castle.dta` is a state-year panel: 50 states, 2000–2010, 550 rows. Twenty-one states adopted a castle doctrine
law between 2005 and 2009, at different times: **staggered adoption**, the setting where two-way fixed effects can
make forbidden comparisons (Topic 11). The variables the design uses:

| variable | what it is |
|---|---|
| `l_homicide` | log of homicides per 100,000 people (the outcome); other `l_*` columns are other crimes |
| `cdl` | the share of the year a castle doctrine law was in effect: between 0 and 1 in the adoption year, 1 after |
| `effyear` | the year the law took effect (missing for the 29 states that never adopted) |
| `popwt` | the population weight the paper uses |
| `sid`, `state`, `year` | state and year identifiers |
| `r20001`–`r20104` | region-by-year indicators (Census region times year) |
| `lead1`–`lead9`, `lag0`–`lag5` | event-time indicators: years before and after adoption |

The adoption year is partly treated (`cdl` between 0 and 1). How the paper codes it, and how you code it, is a
Section 2 decision to document.

In [ ]:
# GIVEN — Section 2 starts here: load the data
castle = pd.read_stata("https://raw.githubusercontent.com/scunning1975/mixtape/master/castle.dta")
print(f"{castle['sid'].nunique()} states, {castle['year'].min()}-{castle['year'].max()}, "
      f"{castle.groupby('sid')['effyear'].first().notna().sum()} adopters")
castle.groupby("sid")["effyear"].first().value_counts().sort_index().rename("states adopting")

## Section 2: direct replication — YOUR TURN, no AI

Estimate the paper's headline two-way fixed effects regression: `l_homicide` on `cdl` with state and year fixed
effects, plus whatever else the paper's table includes (its region-by-year effects, its controls), with the
paper's weights and clustering. `smf.wls(formula, data, weights=...).fit(cov_type="cluster", cov_kwds={"groups": ...})`
fits a weighted regression with clustered standard errors; `C(sid)` in a formula adds state fixed effects.

Set these names, because Section 3 reuses them:

- `OUTCOME`: the outcome column (`"l_homicide"` for the headline); `WEIGHTED`: `True` if you use `popwt`
- `twfe_fit`: your fitted result; `ESTIMATE`: your coefficient on `cdl`, as a float

In [ ]:
# YOUR TURN — Section 2, no AI: set OUTCOME and WEIGHTED, fit twfe_fit, and set ESTIMATE
OUTCOME = None
WEIGHTED = None
twfe_fit = None
ESTIMATE = None

### Your number, with its unit

`l_homicide` is a log, so the coefficient on `cdl` is a change in the log homicide rate. Times 100 it is in **log
points** (0.05 is 5 log points), approximately the percent change in the rate; 100·(e^b − 1) is the exact percent,
and both readings pass under "percent". It is not a change in
"homicides", and not percentage points. Fill all five names:

- `PRINTED`: the coefficient exactly as your code prints it
- `IN_UNITS`: the same estimate in the unit you name in `UNIT`
- `UNIT`: one of `log points`, `percent`
- `PAPER_NUMBER`, `PAPER_WHERE`: the paper's own estimate, and its table, column and row, read from the paper

In [ ]:
# YOUR TURN — Section 2, no AI
PRINTED = None
IN_UNITS = None
UNIT = None
PAPER_NUMBER = None
PAPER_WHERE = None

In [ ]:
# CHECK — Section 2: your headline number, with its unit. Run it; you need not read it.
def check_headline(printed, in_units, unit, paper_number, paper_where, units_ok):
    """Is the unit right for this outcome, and is the in-unit number the printed one converted?"""
    if any(v is None for v in (printed, in_units, unit, paper_number, paper_where)):
        print("Not filled yet: set all five names in the cell above.")
        return False
    u = " ".join(str(unit).lower().split())
    if u not in units_ok:
        print(f"'{unit}' is not a unit this outcome is measured in. Allowed here: " + "; ".join(sorted(units_ok)) + ".")
        print("Ask what the outcome variable is: a log, a proportion (0 to 1), or a rate per 100,000.")
        return False
    wants = [f(float(printed)) for f in units_ok[u]]     # every accepted reading of this unit
    if not any(abs(float(in_units) - w) <= max(0.005, 0.02 * abs(w)) for w in wants):
        print(f"Your in-unit number does not follow from your printed estimate in '{unit}'.")
        print("Convert the printed coefficient again: what does one unit of the outcome mean here?")
        return False
    if not str(paper_where).strip() or len(str(paper_where).split()) < 2:
        print("Name where the paper's number is: table, column (or figure), and row.")
        return False
    print("Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.")
    return True

CH_UNITS = {"log points": [lambda b: 100 * b], "percent": [lambda b: 100 * b, lambda b: 100 * (np.exp(b) - 1)]}
ok = check_headline(PRINTED, IN_UNITS, UNIT, PAPER_NUMBER, PAPER_WHERE, CH_UNITS)

## Section 3: robustness — GIVEN code, AI allowed from here

These checks take your Section 2 outcome and weighting. For each, write what it implies about the original claim.

**1. Estimators that make only clean comparisons.** With staggered adoption, two-way fixed effects also compares
late adopters with states that were treated earlier (Topic 11's forbidden comparison). The function below builds
Callaway and Sant'Anna's group-time effects by hand, as two-by-two differences: for each adoption cohort g and year
t, the cohort's change since the year before adoption minus the same change in a control group that is untreated at
t, either the **never-treated** states or the **not-yet-treated** ones. Each two-by-two uses population weights if
you did. It has no covariates and **no region-by-year effects**, so it answers a different question from a TWFE
that has them: before you read a placebo outcome through it, run your TWFE with and without the region-by-year
effects and see which comparison the placebo result follows.

In [ ]:
# GIVEN — Section 3: Callaway–Sant'Anna group-time effects by hand, and their averages
def att_gt(outcome, weighted, control="never", first="effyear", data=None):
    d = (castle if data is None else data).copy()
    g_of = d.groupby("sid")["effyear"].first()
    if first == "full":                         # count the first FULL year as the start of treatment
        g_of = g_of + 1
    d["g"] = d["sid"].map(g_of)
    w = d["popwt"] if weighted else pd.Series(1.0, index=d.index)
    d["w"] = w

    def wmean(rows):
        return np.average(rows[outcome], weights=rows["w"])

    cells = []
    for g in sorted(g_of.dropna().unique()):
        coh = d[d["g"] == g]
        base = g - 1
        for t in sorted(d["year"].unique()):
            if t == base:
                continue
            if control == "never":
                ctl = d[d["g"].isna()]
            else:                                   # not yet treated at t, and not yet at the base year either
                ctl = d[d["g"].isna() | (d["g"] > max(t, base))]
            ctl = ctl[ctl["g"] != g] if control != "never" else ctl
            a = wmean(coh[coh.year == t]) - wmean(coh[coh.year == base])
            b = wmean(ctl[ctl.year == t]) - wmean(ctl[ctl.year == base])
            cells.append({"g": int(g), "t": int(t), "e": int(t - g), "att": a - b,
                          "size": coh[coh.year == t]["w"].sum() if weighted else coh["sid"].nunique()})
    cells = pd.DataFrame(cells)
    post = cells[cells["e"] >= 0]
    overall = np.average(post["att"], weights=post["size"])
    by_e = cells.groupby("e").apply(lambda x: np.average(x["att"], weights=x["size"]), include_groups=False)
    return overall, by_e, cells


def state_bootstrap(outcome, weighted, control, reps=200, seed=42):
    rng = np.random.default_rng(seed)
    states = castle["sid"].unique()
    draws = []
    for _ in range(reps):
        pick = rng.choice(states, size=len(states), replace=True)
        boot = pd.concat([castle[castle["sid"] == s].assign(sid=i) for i, s in enumerate(pick)])
        try:
            draws.append(att_gt(outcome, weighted, control, data=boot)[0])
        except (ZeroDivisionError, ValueError):
            continue
    return np.std(draws, ddof=1)


if OUTCOME is None:
    print("Fill Section 2 first: OUTCOME is not set yet.")
else:
    for control in ("never", "notyet"):
        overall, by_e, _ = att_gt(OUTCOME, WEIGHTED, control)
        se = state_bootstrap(OUTCOME, WEIGHTED, control)
        print(f"CS by hand, {control}-treated controls: {overall:+.4f} (state bootstrap SE {se:.4f})")
    sy = smf.wls(f"{OUTCOME} ~ cdl + C(sid) + C(year)", data=castle, weights=castle["popwt"] if WEIGHTED else None
                 ).fit(cov_type="cluster", cov_kwds={"groups": castle["sid"]}) if WEIGHTED else smf.ols(
                 f"{OUTCOME} ~ cdl + C(sid) + C(year)", data=castle).fit(cov_type="cluster", cov_kwds={"groups": castle["sid"]})
    print(f"TWFE, state and year effects only:       {sy.params['cdl']:+.4f} (SE {sy.bse['cdl']:.4f})   <- same fixed effects as CS")
    print(f"Your Section 2 TWFE:                     {ESTIMATE:+.4f} (SE {twfe_fit.bse['cdl']:.4f})")

**2. Event time.** The same group-time effects averaged by years since adoption, never-treated controls. Before
adoption (e < 0) they should hover around zero if trends were parallel; look at the pattern, not one cell. In
this file the years before adoption sit a few log points above the reference year under both estimators: decide
whether that is a pre-trend, noise, or a sign the reference year is unusual, and say what it does to the claim. Beside
it, the TWFE event study built from the file's lead and lag indicators (the year before adoption is the reference).

In [ ]:
# GIVEN — Section 3: event-time averages, and the TWFE event study
if OUTCOME is None:
    print("Fill Section 2 first: OUTCOME is not set yet.")
else:
    _, by_e, _ = att_gt(OUTCOME, WEIGHTED, "never")
    leads = [c for c in castle.columns if c.startswith("lead") and c != "lead1" and castle[c].sum() > 0]
    lags = [c for c in castle.columns if c.startswith("lag") and c[3:].isdigit() and castle[c].sum() > 0]
    w = castle["popwt"] if WEIGHTED else None
    fit = (smf.wls if WEIGHTED else smf.ols)(f"{OUTCOME} ~ {' + '.join(leads + lags)} + C(sid) + C(year)", data=castle,
                                             **({"weights": w} if WEIGHTED else {})
                                             ).fit(cov_type="cluster", cov_kwds={"groups": castle["sid"]})
    es = {-int(c[4:]): fit.params[c] for c in leads}
    es.update({int(c[3:]): fit.params[c] for c in lags})
    table = pd.DataFrame({"CS by hand": by_e, "TWFE event study": pd.Series(es)}).loc[-5:5].round(4)
    print(table.to_string())

**3. Your own specification.** Change at least one choice and justify it: another outcome (a crime the law should
not move, such as `l_burglary`, is a placebo), the control group, weights or no weights, counting treatment from the
first full year, or a sample (dropping Florida, the first adopter, or the South). `MY_SPEC` is a dictionary:
`{"outcome": a column, "control": "never" or "notyet", "weighted": True or False, "first": "effyear" or "full",
"sample": None or a pandas query string}`.

In [ ]:
# YOUR TURN — Section 3: your specification, and two sentences on why it is defensible
MY_SPEC = None
MY_SPEC_WHY = None

In [ ]:
# CHECK — Section 3: your own specification. Run it; you need not read it.
def check_my_spec(my_spec, baseline, why):
    if my_spec is None or why is None:
        print("Not filled yet: set MY_SPEC and MY_SPEC_WHY above.")
        return False
    if my_spec == baseline:
        print("MY_SPEC is the baseline. Change at least one choice, and say why that choice is defensible.")
        return False
    sentences = [s for s in str(why).replace("?", ".").split(".") if len(s.split()) >= 5]
    if len(sentences) < 2:
        print("MY_SPEC_WHY needs at least two sentences: what you changed, and why a careful author might have made that choice.")
        return False
    print("Specification accepted. Report it beside the baseline and say what the difference implies about the original claim.")
    return True

BASELINE = None if OUTCOME is None else {"outcome": OUTCOME, "control": "never", "weighted": WEIGHTED, "first": "effyear", "sample": None}
if check_my_spec(MY_SPEC, BASELINE, MY_SPEC_WHY):
    d = castle.query(MY_SPEC["sample"]) if MY_SPEC["sample"] else castle
    overall, _, _ = att_gt(MY_SPEC["outcome"], MY_SPEC["weighted"], MY_SPEC["control"], MY_SPEC["first"], data=d)
    print(f"Your specification (CS by hand): {overall:+.4f}")
    base, _, _ = att_gt(OUTCOME, WEIGHTED, "never")
    print(f"Baseline (CS by hand, never-treated): {base:+.4f}; Section 2 TWFE {ESTIMATE:+.4f}")

**4. Your other exercises.** Section 3 asks for at least four. Natural ones here: a Goodman-Bacon decomposition of
your TWFE (which comparisons carry the weight); the same estimates without the region-by-year effects; inference
with few treated clusters (21 adopters), for example a wild cluster bootstrap; and the other crimes the paper
studies. Write them below this cell.

## Sections 4 to 6, in your repository

- **Section 4, does the finding hold?** Use your Section 3 results. "Holds in a narrower population", "holds at a
  smaller magnitude" and "holds under an assumption the authors did not state" are all answers.
- **Section 5, peer audit.** Assigned on the date the brief gives, never on your own paper; file the issue by the
  project deadline in the brief's Checkpoints list.
- **Section 6, your own repository.** Move the code you keep from this notebook into the template's layout, pin
  versions in `requirements.txt` (this notebook was built with pandas 2.2.2, numpy 2.0.2, statsmodels 0.14.6,
  linearmodels 7.1, rdrobust 2.0.0 and rddensity 3.0), and make the README name the one command that reproduces
  your headline table. Then clone your own repository into an empty folder and run that command.